In [1]:
from google.colab import files
uploaded = files.upload()

Saving sales_analysis_project.zip to sales_analysis_project.zip


In [2]:
import zipfile
from pathlib import Path

zip_name = next(name for name in uploaded if name.lower().endswith(".zip"))

with zipfile.ZipFile(zip_name, "r") as project_zip:
    project_zip.extractall("sales_project")

pipelines = list(Path("sales_project").rglob("pipeline.py"))
project_folder = pipelines[0].parent

print("Project ready!")
print("Project folder:", project_folder)

Project ready!
Project folder: sales_project/sales_analysis_project


In [3]:
import pandas as pd

data = [
    ["ORD001", "2026-01-05", "Keyboard", 2, 800, "C001", "Accessories", "West"],
    ["ORD001", "2026-01-05", "Mouse", 1, 400, "C001", "Accessories", "West"],
    ["ORD002", "2026-01-12", "Monitor", 1, 9000, "C002", "Electronics", "North"],
    ["ORD003", "2026-02-03", "Mouse", 3, 400, "C001", "Accessories", "West"],
    ["ORD004", "2026-02-15", "Keyboard", 1, 800, "C003", "Accessories", "South"],
    ["ORD005", "2026-03-02", "Monitor", 2, 9000, "C002", "Electronics", "North"],
    ["ORD006", "2026-03-10", "Headphones", 2, 1500, "C004", "Electronics", "East"],
    ["ORD007", "wrong-date", "Mouse", 1, 400, "C005", "Accessories", "West"],
]

columns = [
    "order_id", "order_date", "product", "quantity",
    "unit_price", "customer_id", "category", "region"
]

sales = pd.DataFrame(data, columns=columns)
csv_path = project_folder / "practice_sales.csv"
sales.to_csv(csv_path, index=False)

print("Practice sales CSV ready!")
display(sales)

Practice sales CSV ready!


,order_id,order_date,product,quantity,unit_price,customer_id,category,region
0,ORD001,2026-01-05,Keyboard,2,800,C001,Accessories,West
1,ORD001,2026-01-05,Mouse,1,400,C001,Accessories,West
2,ORD002,2026-01-12,Monitor,1,9000,C002,Electronics,North
3,ORD003,2026-02-03,Mouse,3,400,C001,Accessories,West
4,ORD004,2026-02-15,Keyboard,1,800,C003,Accessories,South
5,ORD005,2026-03-02,Monitor,2,9000,C002,Electronics,North
6,ORD006,2026-03-10,Headphones,2,1500,C004,Electronics,East
7,ORD007,wrong-date,Mouse,1,400,C005,Accessories,West


In [4]:
import subprocess
import sys

output_folder = project_folder / "practice_results"

result = subprocess.run(
    [
        sys.executable,
        str(project_folder / "pipeline.py"),
        "--input", str(csv_path),
        "--date-format", "%Y-%m-%d",
        "--output", str(output_folder)
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Analysis complete!")
    print(result.stdout)
else:
    print("Error aaya:")
    print(result.stderr)

Analysis complete!
{
  "input_rows": 8,
  "accepted_rows": 7,
  "rejected_rows": 1,
  "suspected_duplicate_rows_retained": 0,
  "missing_customer_rows": 0,
  "sales_revenue": 34000.0
}



In [5]:
import pandas as pd

result_files = sorted(
    (output_folder / "query_results").glob("01_*.csv")
)

if result_files:
    summary = pd.read_csv(result_files[0])
    display(summary)
else:
    print("Result file nahi mili. Pichhle step ka output check karo.")

,orders,identified_customers,units,revenue,average_order_value
0,6,4,12,34000.0,5666.67


In [6]:
from IPython.display import display, Markdown
import pandas as pd

result_files = sorted(
    (output_folder / "query_results").glob("*.csv")
)

if not result_files:
    print("Results nahi mile. Analysis wala step dobara check karo.")
else:
    for file in result_files:
        title = file.stem.replace("_", " ").title()
        display(Markdown(f"### {title}"))
        display(pd.read_csv(file))

### 01 Executive Kpis

,orders,identified_customers,units,revenue,average_order_value
0,6,4,12,34000.0,5666.67


### 02 Monthly Growth

,month,revenue,mom_growth_pct
0,2026-01,11000.0,NaN
1,2026-02,2000.0,-81.82
2,2026-03,21000.0,950.00


### 03 Top Products

,product,units,revenue
0,Monitor,3,27000.0
1,Headphones,2,3000.0
2,Keyboard,3,2400.0
3,Mouse,4,1600.0


... (10 outputs left out)

### 09 Basket Size

,units,orders,average_order_value
0,1,2,4900.0
1,2,2,10500.0
2,3,2,1600.0


### 10 Product Concentration

,product,revenue,share_pct,cumulative_share_pct
0,Monitor,27000.0,79.41,79.41
1,Headphones,3000.0,8.82,88.24
2,Keyboard,2400.0,7.06,95.29
3,Mouse,1600.0,4.71,100.00


In [8]:
import shutil
from google.colab import files

results_zip = shutil.make_archive(
    "/content/practice_sales_results",
    "zip",
    root_dir=str(output_folder)
)

files.download(results_zip)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>